# STEP 9-A — 1단계 입력 비교: full vs f320

**1단계가 `full`(강아지 전신 사진)로 학습했는데, 배포에서 보호자는 촬영
가이드대로 병변에 다가가서 찍습니다.** STEP 8 에서 열어둔 위험입니다.

## 왜 f320 인가

| | full | f320 |
|---|---|---|
| 창 크기 | 화면 전체(중앙 정사각) | 항상 320px 고정 |
| 구도 | 전신 | 근접 — 배포와 비슷 |
| 창 크기가 정답을 흘리나 | 안 흘림 (bbox 안 씀) | 안 흘림 (창이 병변 크기와 무관) |

⚠️ STEP 4A 에서 "f320 불필요" 로 닫았던 카드입니다. 그때 근거는 **2단계
기준**(지름길 없음·배율 견고성에 도움 안 됨)이었습니다. 지금 묻는 건
**1단계의 분포 어긋남**이라 질문이 다릅니다.

## 이 노트북이 하는 것

모델·증강은 STEP 6·7 이 이미 정한 대로 고정합니다 (`effnetv2_s` +
`photometric`). **입력만** `full` ↔ `f320` 으로 바꿔 비교합니다.
2단계는 건드리지 않습니다.

## 판정 기준 (실험 **전에** 못 박습니다 — 작업 규칙 2)

1. AUROC 가 잡음(±0.01) 밖으로 떨어지지 않아야 후보
2. 흐림 하락이 5%p 넘게 나빠지면 탈락 — f320 이 화질 지름길을 다시 열면 안 됨
3. 스크리닝 recall 은 여기서 안 봅니다 (임계값은 val 로 별도로 잡는 값)

규칙은 `experiments.stage1_crop_report()` 에 있습니다.

> ⚠️ **holdout 은 여기서 안 봅니다.** 후보를 고르는 데 holdout 을 쓰면 그 순간
> 오염됩니다. 05 가 엽니다.

## 필요한 Kaggle 입력

`dogskin-full`, `dogskin-f320` — 이번엔 `dogskin-m25` 는 필요 없습니다
(2단계는 안 돌립니다).

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

_PKGS = ["timm", "imagehash", "pyarrow", "grad-cam", "albumentations"]
_ok = False
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"],
                  check=False).returncode == 0:
    _ok = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q",
                          "--system", *_PKGS], check=False).returncode == 0
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-23.6"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 0-b. 로컬에서 만든 데이터 불러오기

In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ── 다른 환경에서 학습한 체크포인트 가져오기 (Colab → Kaggle 이주) ──────
#    Colab 에서 이미 학습을 끝냈다면, Drive 의 dogskin_work/checkpoints 를
#    Kaggle 데이터셋으로 올린 뒤 그 경로를 여기에 주세요.
#    가져온 실험은 '완료' 로 인식되어 학습 셀이 ⏭️ 로 건너뜁니다.
#
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
# ── 1. 설정 ─────────────────────────────────────────────────────
import torch
from src import labels, split, crop, experiments, stages
from src.config import CLASSES_STAGE1

env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE  = 384          # 03 에서 실측으로 채택
EPOCHS    = 12           # 후보 거르기용 (STEP 6 과 같은 조건)
SUBSET    = 0.55         # 학습셋만. 검증셋은 그대로
N_ROBUST  = 2000
MODEL     = "effnetv2_s"     # STEP 6·7 이 채택
AUG       = "photometric"    # STEP 6·7 이 채택
CROPS     = ("full", "f320")

# ── 시작 전에 전부 확인합니다 (03c 에서 배운 것 — 학습 루프 안에서 확인하면
#    78분 뒤에 터집니다) ──────────────────────────────────────────
have = crop.available_tags()
print(f"사용 가능한 크롭 태그: {have}")
missing = [c for c in CROPS if c not in have]
if missing:
    raise SystemExit(
        f"❌ 크롭 {missing} 이 없습니다. 붙어 있는 것: {have}\n"
        "   Kaggle 우측 [Add Input] 에서 dogskin-full / dogskin-f320 을 붙이세요.")

df = labels.load(env.work_root() / "manifests" / "manifest_final.parquet")
print(f"매니페스트 {len(df):,}행 / 개체 {df['animal_id'].nunique():,}마리")
print(f"모델 {MODEL} / 증강 {AUG} 고정 · 크롭 {CROPS} 비교 · {IMG_SIZE}px · {EPOCHS}에폭")

In [ ]:
# ── 2. 시작 전에 시간부터 재봅니다 (작업 규칙 3) ────────────────
experiments.estimate_runtime([MODEL] * len(CROPS), IMG_SIZE,
                             n_train=int(len(df) * 0.5 * SUBSET), epochs=EPOCHS,
                             n_conditions=len(CROPS), device=DEV)
print("\n⚠️ Kaggle 세션 한도(9시간)를 넘길 것 같으면 위에서 멈추고 다시 추정하세요.")
print("   [Save Version] → Save & Run All (Commit) 으로 돌려야 브라우저를 닫아도 남습니다.")

In [ ]:
# ── 3. 스윕 ─────────────────────────────────────────────────────
# ★ 이어받기: 끝난 조건은 train.fit 이 건너뜁니다. 세션이 죽으면 이 셀만 다시 돌리세요.
runs = []
for tag in CROPS:
    try:
        d = crop.switch_tag(df, tag)      # 커버리지 95% 미만이면 여기서 멈춥니다
        view = stages.to_stage1(d)
        split.verify(view, fold=0, strict=True)   # 누수가 있으면 여기서 에러

        r = experiments.train_and_measure(
            view, stage=1, img_size=IMG_SIZE, crop_tag=tag, device=DEV,
            epochs=EPOCHS, model_name=MODEL, aug=AUG,
            subset_frac=SUBSET, n_robust=N_ROBUST,
            # 배율 교란은 1단계에서 이미 통과했습니다 (STEP 4A). 궁금한 건
            # 화질 지름길이 다시 열리는지라 그쪽만 잽니다.
            measure_robust=False, measure_blur=True)
        runs.append(r)
    except torch.cuda.OutOfMemoryError:
        print(f"❌ {tag}: VRAM 부족 — 건너뜁니다")
    except Exception as exc:                                    # noqa: BLE001
        print(f"❌ {tag}: {type(exc).__name__}: {exc}")
    finally:
        import gc
        gc.collect()
        if DEV == "cuda":
            torch.cuda.empty_cache()

if not runs:
    raise SystemExit("❌ 성공한 실행이 0개입니다. 위 오류 메시지를 확인하세요.")
if not any(r["crop_tag"] == "full" for r in runs):
    print("⚠️ 기준 full 이 실패했습니다 — 상대 비교가 약해집니다.")
print(f"\n✅ {len(runs)}/{len(CROPS)}개 완료")

## 4. 판정

규칙은 `src/experiments.py` 에 있습니다.

In [ ]:
verdict = experiments.stage1_crop_report(runs, base_crop="full")

In [ ]:
# ── 5. 기록 남기기 ──────────────────────────────────────────────
import json

out = env.work_root() / "reports"
out.mkdir(parents=True, exist_ok=True)
payload = {
    "step": "STEP9A_1단계_f320비교",
    "model": MODEL, "aug": AUG, "img_size": IMG_SIZE, "epochs": EPOCHS,
    "subset_frac": SUBSET, "n_robust": N_ROBUST,
    "runs": [{k: v for k, v in r.items()
              if isinstance(v, (str, int, float, bool, type(None)))} for r in runs],
    "picked": verdict.get("best", {}).get("crop_tag"),
}
(out / "step9a_stage1_crop.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", out / "step9a_stage1_crop.json")
print("\n다음 —")
print(f"  1) 채택 후보 '{payload['picked']}' 로 03 을 풀 데이터·풀 에폭 재학습")
print("  2) 그 release 를 05 에 붙여 holdout 한 번")
print("  ⚠️ 여기 숫자는 서브셋 결과입니다. 보고서에는 풀 학습 숫자를 쓰세요.")

---

## 관련 기록

- [`docs/results/STEP8_1단계교체_holdout_실측.md`](../docs/results/STEP8_1단계교체_holdout_실측.md) — 이 실험의 동기
- [`docs/results/STEP6_1단계_2x2_실측.md`](../docs/results/STEP6_1단계_2x2_실측.md) — 백본·증강을 먼저 정한 근거
- [`docs/results/STEP4A_베이스라인_실측.md`](../docs/results/STEP4A_베이스라인_실측.md) — f320 을 "불필요" 로 닫았던 2단계 기준 결론 (질문이 다름)